<h2>Bit-Flipping (BF) LDPC decoder</h2>
 
<b>Implements the hard-decision iterative decoding steps:
  Step 1: Hard-decision demodulation (continuous -> binary)
  Step 2: Initialization
  Step 3: Iterative loop (syndrome -> unsatisfied-check counts -> flip bits)
  Step 4: Final output and message extraction</b>
 
<b>Reuses the encoder / modulation / channel functions from ldpc_encoder.py
rather than redefining them.</b>

In [12]:
%run ldpc_encoder.ipynb

N=6, K=4, z=2  ->  base matrix B (shape (1, 3)):
[[1 1 1]] 

Parity-check matrix H (lifted from base matrix):
[[0 1 0 1 0 1]
 [1 0 1 0 1 0]] 

Systematic parity-check matrix H_sys = [P | I]:
[[1 0 1 0 1 0]
 [0 1 0 1 0 1]] 

Generator matrix G = [I | P^T]:
[[1 0 0 0 1 0]
 [0 1 0 0 0 1]
 [0 0 1 0 1 0]
 [0 0 0 1 0 1]] 

Message u = [0, 1, 1, 1]
Codeword v = [0 1 1 1 1 0]

BPSK symbols x        = [ 1. -1. -1. -1. -1.  1.]
Noise std (sigma)     = 0.6131  (Eb/N0 = 3.0 dB, rate = 0.6666666666666666)
Received samples r    = [ 0.0525 -1.2143 -1.2487 -2.223  -0.5053  0.518 ]

Hard-decision bits    = [0 1 1 1 1 0]
(compare to original codeword v = [0 1 1 1 1 0])
Verifying all 16 possible 4-bit messages give valid codewords (H.v^T = 0):
  u=[0, 0, 0, 0] -> v=[0 0 0 0 0 0]  syndrome=[0 0]  OK
  u=[0, 0, 0, 1] -> v=[0 0 0 1 0 1]  syndrome=[0 0]  OK
  u=[0, 0, 1, 0] -> v=[0 0 1 0 1 0]  syndrome=[0 0]  OK
  u=[0, 0, 1, 1] -> v=[0 0 1 1 1 1]  syndrome=[0 0]  OK
  u=[0, 1, 0, 0] -> v=[0 1 0 0 0 1]  synd

#### STEP 1: Hard-Decision Demodulation

In [13]:
def hard_decision_demodulate(y):
    """
    Convert continuous received samples y into a binary hard-decision
    vector z, following the BPSK mapping used at the transmitter
    (0 -> +1, 1 -> -1):
        y_n > 0  ->  z_n = 0
        y_n <= 0 ->  z_n = 1
    """
    y = np.asarray(y, dtype=float)
    z = np.where(y > 0, 0, 1).astype(int)
    return z

#### STEP 3A: Compute the Syndrome

In [14]:
def compute_syndrome(z, H):
    """s = z . H^T (mod 2). s_m = 1 means the m-th parity check is unsatisfied."""
    return (H @ z) % 2

#### STEP 3B: Count Unsatisfied Checks per bit (f_n)

In [15]:
def count_unsatisfied_checks(H, s):
    """
    f_n = sum over all rows m where H[m, n] == 1 of s_m.
    Vectorized as H^T . s (ordinary integer sum, not mod 2).
    """
    f = H.T @ s
    return f

#### STEP 3C / 3D + full iterative loop

In [16]:
def bit_flipping_decode(y, H, max_iterations=50, rule="max", threshold=None,
                         tie_break="all", rng=None, verbose=False):
    """
    Bit-Flipping decoder.
 
    Parameters
    ----------
    y : array of real-valued received samples (from BPSK + AWGN)
    H : parity-check matrix (m x n)
    max_iterations : cap on iterations to avoid infinite looping
    rule : "max"       -> Rule 2: flip the bit(s) with the largest f_n
           "threshold" -> Rule 1: flip every bit with f_n > threshold
    threshold : required design threshold T if rule="threshold"
    tie_break : how to handle multiple bits tied for the max f_n (rule="max"):
                "all"    -> flip every tied bit each iteration (matches the
                            algorithm description literally, but can OSCILLATE
                            forever on small/low-degree codes, since flipping
                            several bits at once can undo a correct flip with
                            an incorrect one on the next pass)
                "random" -> flip a single, randomly-chosen bit among the ties
                            (standard practical fix to avoid oscillation)
    rng : numpy Generator used when tie_break="random" (created if None)
    verbose : print per-iteration diagnostics
 
    Returns
    -------
    z : decoded 0/1 codeword (best estimate)
    n_iterations : number of iterations actually run
    success : True if a valid codeword (syndrome all-zero) was found
    """
    if rule == "threshold" and threshold is None:
        raise ValueError("threshold rule selected but no `threshold` value given.")
    if tie_break == "random" and rng is None:
        rng = np.random.default_rng()
 
    # ---- Step 1: Hard-decision demodulation ----
    z = hard_decision_demodulate(y)
 
    # ---- Step 2: Initialization ----
    # (z already set above; max_iterations is the loop cap)
 
    for iteration in range(1, max_iterations + 1):
        # ---- Step 3A: Compute the syndrome ----
        s = compute_syndrome(z, H)
 
        if not s.any():
            if verbose:
                print(f"  Converged after {iteration - 1} iteration(s).")
            return z, iteration - 1, True
 
        # ---- Step 3B: Count unsatisfied checks per bit ----
        f = count_unsatisfied_checks(H, s)
 
        # ---- Step 3C: Identify bits to flip ----
        if rule == "max":
            flip_indices = np.flatnonzero(f == f.max())
            if tie_break == "random" and flip_indices.size > 1:
                flip_indices = np.array([rng.choice(flip_indices)])
        elif rule == "threshold":
            flip_indices = np.flatnonzero(f > threshold)
            if flip_indices.size == 0:
                # Nobody crosses the threshold but syndrome isn't zero yet:
                # fall back to flipping the worst offender(s) to avoid stalling.
                flip_indices = np.flatnonzero(f == f.max())
                if tie_break == "random" and flip_indices.size > 1:
                    flip_indices = np.array([rng.choice(flip_indices)])
        else:
            raise ValueError("rule must be 'max' or 'threshold'")
 
        if verbose:
            print(f"  Iter {iteration}: syndrome weight={s.sum()}, "
                  f"f={f}, flipping bits={flip_indices.tolist()}")
 
        # ---- Step 3D: Execute the flip ----
        z[flip_indices] ^= 1
 
    # Loop ended without converging inside max_iterations; final check
    s = compute_syndrome(z, H)
    success = not s.any()
    return z, max_iterations, success

#### STEP 4: Extract message bits from the decoded (systematic) codeword

In [17]:
def extract_message_bits(z_hat, K):
    """Since encoding is systematic, the first K bits are the message."""
    return z_hat[:K]

### Full pipeline

In [18]:
# ---- Change these to experiment ----
N = 6
K = 3
z_lift = 3          # lifting factor (named z_lift to avoid clashing
                     # with the decoded vector `z` used below)
ebno_db = 2.0
max_iterations = 20

In [19]:
# Build the code (reusing the encoder module)
B = generate_base_matrix(N, K, z_lift, seed=0)
H = build_parity_check_matrix(B, z_lift)
H_sys, col_order = to_systematic_form(H)
G = derive_generator_matrix(H_sys, K)

print(f"N={N}, K={K}, z={z_lift}")
print("H_sys =\n", H_sys, "\n")

N=6, K=3, z=3
H_sys =
 [[0 1 0 1 0 0]
 [0 0 1 0 1 0]
 [1 0 0 0 0 1]] 



In [20]:
# Random K-bit message -> encode -> modulate -> AWGN
rng = np.random.default_rng(7)
#u = rng.integers(0, 2, size=K).tolist()
u = np.array([1, 0, 1])
v = encode(u, G)
x = bpsk_modulate(v)

R = K / N
sigma = awgn_noise_std(ebno_db, R)
y = awgn_channel(x, sigma)

print(f"Message u        = {u}")
print(f"Codeword v       = {v}")
print(f"Received y       = {np.round(y, 4)}\n")

Message u        = [1 0 1]
Codeword v       = [1 0 1 0 1 1]
Received y       = [-1.357   1.6596  0.0704  1.1042  0.1982 -0.2674]



In [21]:
# ---- Run the Bit-Flipping decoder ----
z_hat, n_iter, success = bit_flipping_decode(
    y, H_sys, max_iterations=max_iterations, rule="max", verbose=True
)

print(f"\nDecoded codeword z_hat = {z_hat}   (converged={success}, iterations={n_iter})")
print(f"Original codeword v    = {v}")

  Converged after 0 iteration(s).

Decoded codeword z_hat = [1 0 0 0 0 1]   (converged=True, iterations=0)
Original codeword v    = [1 0 1 0 1 1]


In [22]:
u_hat = extract_message_bits(z_hat, K)
print(f"\nDecoded message u_hat  = {u_hat.tolist()}")
print(f"Original message u     = {u}")
print("Message recovered correctly!" if list(u_hat) == list(u) else "Message NOT recovered correctly.")


Decoded message u_hat  = [1, 0, 0]
Original message u     = [1 0 1]
Message NOT recovered correctly.
